# Step 5 · Fair comparisons: could luck or word-counting do as well?

**Pain–Comfort Lab · Looking inside a language model** · ⏱️ about 25 minutes · ⚡ GPU recommended

> ### 🤔 Wonder
> A test score of 90% sounds impressive. But **compared to what?** Would a ruler pointing in a random direction do just as well? Would a simple program that only counts words?

Good experiments include **controls**: comparison conditions that test other explanations for a result. In this step you'll build two kinds:

1. **Fake rulers**: 200 built from shuffled labels and 200 pointing in random directions. Does our real ruler clearly beat them?
2. **A word-counter** that never looks inside the language model. Can it solve the task from word clues alone?

## Setup and catch-up

In [ ]:
# 🔧 Setup: just run this cell. It works in Google Colab and in Jupyter on your own computer.
# It makes the lab's helper library (the `pclab` folder) and the sentence files available.
import subprocess, sys
from pathlib import Path
from importlib.metadata import version
from packaging.version import Version

REPO_URL = 'https://github.com/23menon23/pain-comfort-lab'   # the lab's home on GitHub

if Path('../pclab').exists():                    # you are inside a downloaded copy of the repository
    LAB_ROOT = Path('..').resolve()
else:                                            # Google Colab: download the lab once per session
    LAB_ROOT = Path('pain-comfort-lab').resolve()
    if not LAB_ROOT.exists():
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(LAB_ROOT)], check=True)
sys.path.insert(0, str(LAB_ROOT))

if Version(version('transformers')) < Version('4.57'):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers>=4.57'], check=True)
    print('⚠️ Transformers was updated. Choose Runtime → Restart session, then run this cell again.')
else:
    import pclab
    from pclab import look, show_source
    print('✅ Ready. pclab', pclab.__version__, '| Transformers', version('transformers'))

In [ ]:
# 🔧 Load the model: just run this cell (the first download is about 3 GB and takes a few minutes).
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display

# Keep this model for class. 'Qwen/Qwen2.5-0.5B' uses less memory, but its results
# will differ, so record it as a different experiment.
MODEL_NAME = 'Qwen/Qwen2.5-1.5B'

SEED = pclab.set_seed(42)          # fixes random choices so results can be repeated
DEVICE = pclab.device_report()     # 'cuda' (GPU) or 'cpu'
tokenizer, model = pclab.load_model(MODEL_NAME)
LAYERS = pclab.get_layers(model)
N_LAYERS = len(LAYERS)
HIDDEN_SIZE = model.config.hidden_size
pclab.use_lab_style()
print(f"Loaded {pclab.model_facts(model)['model']}: {N_LAYERS} layers, {HIDDEN_SIZE} numbers per token")

In [ ]:
# 🔧 Catch-up: Steps 2–4 (frozen layer, plus the test snapshots)
from sklearn.metrics import accuracy_score, roc_auc_score

state = pclab.rebuild_lab_state(model, tokenizer)
build_df, test_df = state['build_df'], state['test_df']
train_act, y_train = state['train_act'], state['y_train']
unit_vectors, midpoints, BEST_LAYER = state['unit_vectors'], state['midpoints'], state['best_layer']

test_act = pclab.collect(model, tokenizer, test_df['text'], 'Test snapshots')
y_test = test_df['label'].to_numpy()
test_scores = pclab.scores_at_all_layers(test_act, unit_vectors, midpoints)[:, BEST_LAYER]
test_accuracy = accuracy_score(y_test, (test_scores > 0).astype(int))
print(f'Real ruler at layer {BEST_LAYER + 1}: test accuracy {test_accuracy:.1%}')

## Control 1a · Fake rulers from shuffled labels

Within each build pair, we randomly swap which sentence is called "pain" and which "comfort", then build a ruler exactly as before. Everything stays the same **except whether the labels make sense**. We repeat this 200 times.

**✏️ Predict first:** would checking just *one* fake ruler be enough? Why might 200 be better?

In [ ]:
# 🔧 200 shuffled-label rulers, each given the same final test
N_CONTROLS = 200
control_rng = np.random.default_rng(SEED + 1)
X_train = train_act[:, BEST_LAYER, :]          # (32, numbers): build snapshots at the frozen layer
X_test = test_act[:, BEST_LAYER, :]            # (16, numbers)
pair_ids = build_df['pair_id'].to_numpy()

shuffled_accuracies = []
for _ in range(N_CONTROLS):
    fake_labels = y_train.copy()
    for pair in np.unique(pair_ids):
        rows = np.flatnonzero(pair_ids == pair)          # the two row numbers of this pair
        if control_rng.random() < 0.5:                   # a coin flip for each pair...
            fake_labels[rows] = 1 - fake_labels[rows]    # ...swap its two labels (1→0, 0→1)
    pos = X_train[fake_labels == 1].mean(axis=0)
    neg = X_train[fake_labels == 0].mean(axis=0)
    direction = (pos - neg) / max(np.linalg.norm(pos - neg), 1e-8)
    scores = (X_test - (pos + neg) / 2) @ direction
    shuffled_accuracies.append(accuracy_score(y_test, (scores > 0).astype(int)))

shuffled_accuracies = np.array(shuffled_accuracies)
look(shuffled_accuracies, 'shuffled_accuracies', dims=('fake rulers',))

### 🔍 Code walk-through

| Code | What it does | Why |
|---|---|---|
| `np.random.default_rng(SEED + 1)` | a random-number generator with a fixed seed | the same "random" choices every run, so results can be checked. `+ 1` keeps it separate from other random choices in the lab. |
| `np.flatnonzero(pair_ids == pair)` | the row numbers where this pair appears | always two rows: its comfort and pain sentence |
| `1 - fake_labels[rows]` | flips 0↔1 | a quick trick for swapping binary labels |
| swap **within pairs** | each fake "comfort" group still has exactly one sentence from every pair | 🎛️ this keeps the topics balanced. A fully random shuffle could put both sentences of a pair on the same side, which changes more than just the meaning of the labels. |
| `(X_test - midpoint) @ direction` | scores all 16 test sentences at once | `(16, 1536) @ (1536,)` → `(16,)` |
| `shuffled_accuracies.append(...)` then `np.array(...)` | collect in a list, convert at the end | 🧱 a list is easy to grow one item at a time; an array is easy to analyse |

## Control 1b · Fake rulers pointing in random directions

200 arrows pointing in completely random directions, all with length 1, each used with our real midpoint.

In [ ]:
# 🔧 200 random unit directions, scored in one matrix multiplication
random_units = control_rng.normal(size=(N_CONTROLS, HIDDEN_SIZE))
random_units /= np.linalg.norm(random_units, axis=1, keepdims=True)
look(random_units, 'random_units', dims=('fake rulers', 'numbers'))

random_scores = (X_test - midpoints[BEST_LAYER]) @ random_units.T
look(random_scores, 'random_scores', dims=('test sentences', 'fake rulers'))

random_correct = (random_scores > 0).astype(int) == y_test[:, None]
random_accuracies = random_correct.mean(axis=0)
look(random_accuracies, 'random_accuracies', dims=('fake rulers',))

### 🔍 Code walk-through: following the shapes

| Code | Shape | What happens |
|---|---|---|
| `control_rng.normal(size=(200, 1536))` | `(200, 1536)` | 200 random arrows |
| `np.linalg.norm(..., axis=1, keepdims=True)` | `(200, 1)` | each arrow's length. `keepdims=True` keeps the size-1 axis, so the division lines up row by row (the same job `[:, None]` did in Step 3). |
| `random_units.T` | `(1536, 200)` | `.T` **transposes** (flips rows and columns) so the matrix multiplication lines up |
| `(16, 1536) @ (1536, 200)` | `(16, 200)` | every test sentence scored by every fake ruler, in one step |
| `y_test[:, None]` | `(16, 1)` | so each sentence's label is compared with **its own row** of 200 scores |
| `== ...` | `(16, 200)` True/False | correct or not, for every sentence × ruler |
| `.mean(axis=0)` | `(200,)` | average over sentences: one accuracy per fake ruler (True counts as 1) |

Doing 200 rulers in one matrix multiplication instead of a loop isn't just faster; once you can read the shapes, it's also shorter to check.

### How did our real ruler do against the fakes?

In [ ]:
# 🔧 Compare
C = pclab.COLORS
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
bins = (np.arange(len(y_test) + 2) - 0.5) / len(y_test)   # one bar per possible score: 0/16, 1/16, ..., 16/16
for ax, values, title in zip(axes, [shuffled_accuracies, random_accuracies],
                             ['Fake rulers: shuffled labels', 'Fake rulers: random directions']):
    ax.hist(values, bins=bins, color=C['neutral'], alpha=0.8, edgecolor='white', linewidth=2)
    ax.axvline(test_accuracy, color=C['accent'], linewidth=2.5, label=f'Our real ruler ({test_accuracy:.0%})')
    ax.axvline(0.5, color='gray', linestyle=':', label='Chance (50%)')
    ax.set(title=title, xlabel='Final test accuracy', xlim=(-0.05, 1.05))
    ax.legend(fontsize=8)
axes[0].set_ylabel('Number of fake rulers')
plt.tight_layout()
plt.show()

control_summary = pd.DataFrame([
    {'condition': 'Our real ruler', 'mean_accuracy': test_accuracy, 'share_of_fakes_matching_or_beating_it': np.nan},
    {'condition': 'Shuffled labels', 'mean_accuracy': shuffled_accuracies.mean(),
     'share_of_fakes_matching_or_beating_it': (shuffled_accuracies >= test_accuracy).mean()},
    {'condition': 'Random directions', 'mean_accuracy': random_accuracies.mean(),
     'share_of_fakes_matching_or_beating_it': (random_accuracies >= test_accuracy).mean()},
])
display(control_summary.round(3))

**How to read this:** if a large share of fake rulers match or beat ours, we should be cautious. These controls hold the chosen layer fixed, so they're a helpful comparison rather than a formal statistical test.

**🛠️ Tinker:** in the shuffled-label control, some fake rulers can score *worse* than 50%. Why? (Hint: what happens if a fake ruler flips the labels of most pairs?)

## Control 2 · A simple word-counter

This control never looks inside the language model. It counts which words appear in each sentence and learns which words go with pain and which with comfort.

Technically it's **TF–IDF word weights + logistic regression**, a classic text classifier you may have met in an ML course.

- If the word-counter does **as well as** our ruler, the task can be solved with word clues alone. Our ruler *might* still capture something deeper, but this test can't show it. That's an important, honest finding.
- If it does **much worse**, that's more interesting, but still not proof that our ruler captures deep meaning.

**✏️ Predict first:** look back at the test sentences. How well will a word-counter do?

In [ ]:
# 🔧 Build the word-counter from the build sentences only, and give it the same final test
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

word_counter = make_pipeline(
    TfidfVectorizer(ngram_range=(1, 1)),
    LogisticRegression(C=1.0, max_iter=1000, random_state=SEED),
)
word_counter.fit(build_df['text'], y_train)
word_predictions = word_counter.predict(test_df['text'])
word_probabilities = word_counter.predict_proba(test_df['text'])[:, 1]

word_accuracy = accuracy_score(y_test, word_predictions)
print(f'Word-counter test accuracy: {word_accuracy:.1%}   AUC: {roc_auc_score(y_test, word_probabilities):.3f}')
print(f'Our ruler test accuracy:    {test_accuracy:.1%}')

### 🔍 Code walk-through

| Code | What it does | Why |
|---|---|---|
| `TfidfVectorizer(ngram_range=(1, 1))` | turns each sentence into one number per vocabulary word | `(1, 1)` = single words only. TF–IDF gives more weight to words that are frequent in *this* sentence but rare across sentences. |
| `LogisticRegression(C=1.0, ...)` | learns a weight for each word | 🎛️ `C=1.0` is the default amount of regularisation (a brake on extreme weights). Fixed in advance, not tuned on the test set. |
| `make_pipeline(...)` | chains the two steps | so `.fit` and `.predict` run both steps in the right order, and the vectorizer only learns words from the build set |
| `.fit(build_df['text'], y_train)` | learns from the **build** sentences only | same rules as our ruler: no peeking at the test set |
| `.predict_proba(...)[:, 1]` | probability of class 1 (comfort) for each sentence | column 0 is pain, column 1 is comfort; AUC needs a score, not a 0/1 guess |

### 📦 What does the word-counter actually see?

In [ ]:
vectorizer, classifier = word_counter.named_steps['tfidfvectorizer'], word_counter.named_steps['logisticregression']
word_matrix = vectorizer.transform(test_df['text'])
print(type(word_matrix).__name__, '| shape:', word_matrix.shape, '| non-zero entries:', word_matrix.nnz)
print(f'Share of entries that are zero: {1 - word_matrix.nnz / np.prod(word_matrix.shape):.1%}')

> **🧱 Why a *sparse* matrix?** Each row is a sentence and each column is a word in the vocabulary. Any one sentence contains only a handful of those words, so almost every entry is zero. A **sparse matrix** stores only the non-zero entries, which saves a lot of memory when the vocabulary is large. (Compare: our model snapshots are **dense**; nearly every one of the 1,536 numbers is non-zero.)

Now the most revealing part: which words does the word-counter rely on?

In [ ]:
# 🔧 The words with the strongest pull towards comfort and towards pain
weights = pd.DataFrame({'word': vectorizer.get_feature_names_out(), 'weight': classifier.coef_[0]})
strongest = pd.concat([
    weights.nlargest(10, 'weight').assign(pulls_towards='Comfort'),
    weights.nsmallest(10, 'weight').assign(pulls_towards='Pain'),
])
display(strongest.round(3))

These are the **giveaway words**. If they're the same words that appear in the test sentences, the test can be solved from vocabulary alone. That's an honest limitation of this dataset, and an open puzzle: **can you write sentences that the word-counter fails on, but our ruler passes?** That's one of the best investigations in Step 7.

## Save your results

In [ ]:
pclab.save_results(
    'step5_controls',
    tables={'control_summary': control_summary,
            'control_repeats': pd.DataFrame({'shuffled_label_accuracy': shuffled_accuracies,
                                             'random_direction_accuracy': random_accuracies}),
            'word_counter_weights': strongest},
    settings={**pclab.model_facts(model, MODEL_NAME), 'seed': SEED, 'selected_layer_1_based': BEST_LAYER + 1,
              'n_controls': N_CONTROLS, 'test_accuracy': test_accuracy, 'word_counter_accuracy': word_accuracy,
              'control_design': 'fixed selected layer; pairwise label swaps and random unit directions'},
)

---

## 🧭 Explain it

1. How did the real ruler compare with the shuffled-label and random-direction rulers? Use the numbers in your table.
2. How did the word-counter do? What does that tell you, and what does it **not** tell you, about our ruler?
3. Why do we swap labels *within pairs* rather than shuffling all 32 labels freely?
4. Name one other explanation (a **confound**) for our ruler's success that these controls don't rule out.

## 🌱 Curiosity branches

- **More fakes:** run 1,000 random directions instead of 200. How often does a random one match the real ruler?
- **Stability:** reshuffle which pairs go into build, validation and test, five different ways. Does the chosen layer stay the same?
- **Beating the word-counter:** write sentences where pain or comfort is *implied by the situation* (*"She stepped barefoot onto the asphalt at noon in July"*). Step 7 gives you a ready-made scaffold.

**Next:** [Step 6 · From watching to nudging](06_steering.ipynb)